In [1]:
# Detection engine and csv route 

import cv2, csv, os
import numpy as np
from datetime import datetime

CSV_FILE = os.path.abspath("attendance_records.csv")
HEADER = ["date", "time", "class", "source", "detected", "present", "note"]

frontal = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
profile = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_profileface.xml")
print("Models loaded:", not frontal.empty() and not profile.empty())


def iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ix = max(0, min(ax + aw, bx + bw) - max(ax, bx))
    iy = max(0, min(ay + ah, by + bh) - max(ay, by))
    inter = ix * iy
    union = aw * ah + bw * bh - inter
    return inter / union if union else 0


def dedupe(boxes, thresh=0.35):
    keep = []
    for b in sorted(boxes, key=lambda r: r[2] * r[3], reverse=True):
        if all(iou(b, k) < thresh for k in keep):
            keep.append(b)
    return keep


def shrink(frame, max_w=1280):
    h, w = frame.shape[:2]
    if w <= max_w:
        return frame
    return cv2.resize(frame, (max_w, int(h * max_w / w)), interpolation=cv2.INTER_AREA)


def detect_faces(frame, scale=1.1, neighbors=5, min_size=30, use_profile=False, boost=False):
    # Step 1: grayscale + detectMultiScale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if boost:  # helps in poor light
        gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)

    kw = dict(scaleFactor=scale, minNeighbors=neighbors, minSize=(min_size, min_size))
    boxes = [tuple(int(v) for v in r) for r in frontal.detectMultiScale(gray, **kw)]

    if use_profile:  # angled faces: profile + mirrored image for the other side
        boxes += [tuple(int(v) for v in r) for r in profile.detectMultiScale(gray, **kw)]
        w = gray.shape[1]
        for x, y, bw, bh in profile.detectMultiScale(cv2.flip(gray, 1), **kw):
            boxes.append((int(w - x - bw), int(y), int(bw), int(bh)))

    boxes = dedupe(boxes)
    boxes.sort(key=lambda r: (round(r[1] / 80), r[0]))  # number left-to-right, top-to-bottom
    return boxes


def safe(text, limit=120):
    """Stop spreadsheet formula injection (cells starting with = + - @)."""
    text = str(text).strip()[:limit]
    return "'" + text if text[:1] in ("=", "+", "-", "@") else text


def save_record(cls, source, detected, present, note):
    # Step 3: append date, time and counts with the csv module
    is_new = not os.path.exists(CSV_FILE)
    now = datetime.now()
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if is_new:
            w.writerow(HEADER)
        w.writerow([now.strftime("%Y-%m-%d"), now.strftime("%H:%M:%S"),
                    safe(cls, 60), safe(source, 80), detected, present, safe(note, 200)])


def read_records():
    if not os.path.exists(CSV_FILE):
        return []
    with open(CSV_FILE, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

Models loaded: True


In [2]:
# Server route 

from flask import Flask, request, jsonify, send_file, Response

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 25 * 1024 * 1024  # 25 MB


@app.route("/")
def home():
    return PAGE  # defined in Cell 4


@app.route("/detect", methods=["POST"])
def detect():
    f = request.files.get("file")
    if f is None:
        return jsonify(error="No picture received"), 400
    frame = cv2.imdecode(np.frombuffer(f.read(), np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        return jsonify(error="That file is not a readable picture. Use JPG, PNG or WebP."), 400

    frame = shrink(frame)
    try:
        boxes = detect_faces(
            frame,
            scale=float(request.form.get("scale", 1.1)),
            neighbors=int(request.form.get("neighbors", 5)),
            min_size=int(request.form.get("min_size", 30)),
            use_profile=request.form.get("profile") == "1",
            boost=request.form.get("boost") == "1",
        )
    except (ValueError, cv2.error):
        return jsonify(error="Invalid detection settings"), 400
    h, w = frame.shape[:2]
    return jsonify(count=len(boxes), boxes=boxes, width=w, height=h)


@app.route("/save", methods=["POST"])
def save():
    d = request.get_json(force=True, silent=True) or {}
    try:
        detected, present = int(d.get("detected")), int(d.get("confirmed"))
    except (TypeError, ValueError):
        return jsonify(error="Counts must be whole numbers"), 400
    if not (0 <= detected <= 1000 and 0 <= present <= 1000):
        return jsonify(error="Counts must be between 0 and 1000"), 400
    save_record(d.get("cls", ""), d.get("source", ""), detected, present, d.get("note", ""))
    return jsonify(ok=True)


@app.route("/records")
def records():
    return jsonify(rows=list(reversed(read_records()))[:300])


@app.route("/export")
def export():
    if not os.path.exists(CSV_FILE):
        return Response(",".join(HEADER) + "\n", mimetype="text/csv",
                        headers={"Content-Disposition": "attachment; filename=attendance_records.csv"})
    return send_file(CSV_FILE, as_attachment=True, download_name="attendance_records.csv")


@app.route("/clear", methods=["POST"])
def clear():
    if os.path.exists(CSV_FILE):
        os.remove(CSV_FILE)
    return jsonify(ok=True)

In [3]:
# The website

PAGE = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Smart Attendance</title>
<link href="https://fonts.googleapis.com/css2?family=Outfit:wght@500;600;700;800&family=Nunito+Sans:wght@400;600;700&display=swap" rel="stylesheet">
<style>
:root{
  --page:#eef1f8; --ink:#1b1f3b; --muted:#5a6280; --line:#d6dbea;
  --indigo:#4f46e5; --indigo-t:#eef0ff;
  --sky:#0369a1;    --sky-t:#e8f4fc;
  --green:#047857;  --green-t:#e9f8f1;
  --orange:#c2410c; --orange-t:#fff2e8;
  --purple:#7c3aed; --purple-t:#f4efff;
  --danger:#b91c1c; --gold:#ffb000;
}
*{box-sizing:border-box}
body{margin:0;background:var(--page);color:var(--ink);font:400 15px/1.55 "Nunito Sans",system-ui,sans-serif}
h1,h2,h3,.display{font-family:"Outfit",system-ui,sans-serif;margin:0}
.wrap{max-width:1200px;margin:0 auto;padding:0 20px 60px}

/* ---------- top banner ---------- */
.hero{background:linear-gradient(120deg,#1e1b4b 0%,#3730a3 60%,#5b21b6 100%);color:#fff;border-bottom:6px solid var(--gold)}
.hero .wrap{display:flex;justify-content:space-between;align-items:center;gap:16px;flex-wrap:wrap;padding-top:26px;padding-bottom:26px}
.brand{display:flex;gap:14px;align-items:center}
.brand h1{font-size:30px;font-weight:800;letter-spacing:-.01em;line-height:1.1}
.brand p{margin:2px 0 0;color:#cfd3ff}
.clock{background:rgba(255,255,255,.12);border:2px solid rgba(255,255,255,.35);border-radius:12px;padding:10px 16px;font-weight:700;text-align:right}
.clock small{display:block;font-weight:400;color:#cfd3ff}

/* ---------- panels ---------- */
.stack{display:grid;gap:22px;margin-top:24px}
.two{display:grid;grid-template-columns:1fr 1fr;gap:22px}
@media(max-width:900px){.two{grid-template-columns:1fr}}
.panel{--c:var(--indigo);--t:var(--indigo-t);background:#fff;border:3px solid var(--c);border-radius:16px;overflow:hidden;box-shadow:0 4px 0 var(--c)}
.panel>header{background:var(--c);color:#fff;display:flex;align-items:center;gap:12px;padding:12px 18px}
.panel>header h2{font-size:19px;font-weight:700}
.panel>header p{margin:0;font-size:13px;opacity:.9}
.badge{flex:none;width:34px;height:34px;border-radius:50%;background:#fff;color:var(--c);display:grid;place-items:center;font:800 17px "Outfit",sans-serif}
.body{padding:18px;background:var(--t)}
.p-session{--c:var(--indigo);--t:var(--indigo-t)}
.p-camera{--c:var(--sky);--t:var(--sky-t)}
.p-upload{--c:var(--green);--t:var(--green-t)}
.p-result{--c:var(--orange);--t:var(--orange-t)}
.p-records{--c:var(--purple);--t:var(--purple-t)}

/* ---------- form bits ---------- */
label.f{display:block;font-weight:700;margin-bottom:5px}
input[type=text],input[type=number]{width:100%;padding:10px 12px;border:2px solid var(--c);border-radius:10px;background:#fff;font:inherit;color:var(--ink)}
input[type=range]{width:100%;accent-color:var(--c)}
.row2{display:grid;grid-template-columns:1fr 1fr;gap:16px}
@media(max-width:700px){.row2{grid-template-columns:1fr}}
details{margin-top:16px;border:2px dashed var(--c);border-radius:12px;background:#fff}
summary{cursor:pointer;padding:11px 14px;font-weight:700;color:var(--c)}
.setgrid{display:grid;grid-template-columns:repeat(3,1fr);gap:18px;padding:4px 16px 16px}
@media(max-width:700px){.setgrid{grid-template-columns:1fr}}
.setgrid .l{display:flex;justify-content:space-between;font-weight:700}
.setgrid output{color:var(--c)}
.hint{margin:2px 0 0;font-size:13px;color:var(--muted)}
.chk{display:flex;gap:9px;align-items:center;font-weight:600}
.chk input{accent-color:var(--c);width:18px;height:18px}
.setfoot{display:flex;gap:22px;flex-wrap:wrap;align-items:center;padding:0 16px 16px}

/* ---------- buttons ---------- */
.btn{font:700 15px "Nunito Sans",sans-serif;padding:11px 18px;border-radius:10px;border:2px solid var(--c);background:var(--c);color:#fff;cursor:pointer;text-decoration:none;display:inline-block;transition:transform .08s}
.btn:hover:not(:disabled){transform:translateY(-1px);filter:brightness(1.08)}
.btn.ghost{background:#fff;color:var(--c)}
.btn.danger{--c:var(--danger);background:#fff;color:var(--danger)}
.btn:disabled{opacity:.4;cursor:not-allowed}
.btns{display:flex;gap:10px;flex-wrap:wrap;margin-top:14px}
:focus-visible{outline:3px solid var(--gold);outline-offset:2px}

/* ---------- camera and upload areas ---------- */
.screen{position:relative;aspect-ratio:16/9;background:#0b1020;border:3px solid var(--c);border-radius:12px;overflow:hidden;display:grid;place-items:center}
.screen video{position:absolute;inset:0;width:100%;height:100%;object-fit:contain}
.screen .off{color:#b8c4dc;text-align:center;padding:16px}
.screen .off b{display:block;font:700 18px "Outfit",sans-serif;color:#fff}
.screen.flash::after{content:"";position:absolute;inset:0;background:#fff;animation:flash .35s forwards}
@keyframes flash{from{opacity:.9}to{opacity:0}}
.recdot{position:absolute;top:10px;left:12px;background:rgba(0,0,0,.55);color:#fff;border-radius:20px;padding:3px 11px;font-size:13px;font-weight:700}
.recdot::before{content:"";display:inline-block;width:9px;height:9px;border-radius:50%;background:#ef4444;margin-right:7px}
.drop{aspect-ratio:16/9;border:3px dashed var(--c);border-radius:12px;background:#fff;display:grid;place-content:center;gap:8px;text-align:center;padding:16px;cursor:pointer}
.drop.over{background:#d9f4e8;border-style:solid}
.drop svg{margin:0 auto}
.drop strong{font:700 19px "Outfit",sans-serif}
.drop span{color:var(--muted)}

/* ---------- result ---------- */
.resgrid{display:grid;grid-template-columns:minmax(0,1fr) 320px;gap:20px;align-items:start}
@media(max-width:900px){.resgrid{grid-template-columns:1fr}}
.pic{background:#0b1020;border:3px solid var(--c);border-radius:12px;overflow:hidden;min-height:260px;display:grid;place-items:center}
.pic canvas{width:100%;height:auto;display:block}
.pic .none{color:#b8c4dc;text-align:center;padding:20px}
.pic .none b{display:block;font:700 18px "Outfit",sans-serif;color:#fff}
.sidebox{background:#fff;border:3px solid var(--c);border-radius:12px;padding:16px}
.big{font:800 96px/1 "Outfit",sans-serif;color:var(--c);font-variant-numeric:tabular-nums}
.sidebox .cap{font-weight:700;color:var(--muted)}
.status{min-height:22px;margin:6px 0 10px;font-size:14px;color:var(--muted)}
.sidebox hr{border:0;border-top:2px dashed var(--line);margin:14px 0}
.sidebox .btn{width:100%;margin-top:10px}

/* ---------- records ---------- */
.stats{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-bottom:16px}
@media(max-width:700px){.stats{grid-template-columns:1fr}}
.stat{background:#fff;border:3px solid var(--c);border-radius:12px;padding:12px 16px}
.stat b{display:block;font:800 34px "Outfit",sans-serif;color:var(--c);font-variant-numeric:tabular-nums}
.stat span{color:var(--muted);font-weight:600}
.tablewrap{max-height:340px;overflow:auto;border:3px solid var(--c);border-radius:12px;background:#fff}
table{width:100%;border-collapse:collapse;font-size:14px;font-variant-numeric:tabular-nums}
th,td{text-align:left;padding:10px 12px;border-bottom:1px solid var(--line);white-space:nowrap}
td.note{white-space:normal;min-width:140px}
th{position:sticky;top:0;background:var(--c);color:#fff}
tbody tr:nth-child(even){background:#faf8ff}
td.pres{font-weight:800;color:var(--c);text-align:right}
th.pres{text-align:right}
.head{display:flex;justify-content:space-between;gap:10px;flex-wrap:wrap}

#toast{position:fixed;left:50%;bottom:24px;transform:translateX(-50%);background:var(--ink);color:#fff;padding:13px 20px;border-radius:12px;border:2px solid var(--gold);font-weight:700;opacity:0;pointer-events:none;transition:opacity .2s;z-index:20;max-width:92vw}
#toast.show{opacity:1}
#toast.error{background:var(--danger)}
[hidden]{display:none!important}
@media(prefers-reduced-motion:reduce){*{animation:none!important;transition:none!important}}
</style>
</head>
<body>

<div class="hero"><div class="wrap">
  <div class="brand">
    <svg width="54" height="54" viewBox="0 0 34 34" fill="none" stroke="#fff" stroke-width="2.6" stroke-linecap="round" aria-hidden="true">
      <path d="M4 12V6a2 2 0 0 1 2-2h6M22 4h6a2 2 0 0 1 2 2v6M30 22v6a2 2 0 0 1-2 2h-6M12 30H6a2 2 0 0 1-2-2v-6"/>
      <circle cx="17" cy="14" r="3.6" fill="#ffb000" stroke="none"/>
      <path d="M10 25c1.2-4 3.7-5.5 7-5.5s5.8 1.5 7 5.5" stroke="#ffb000"/>
    </svg>
    <div><h1>Smart Attendance</h1><p>Take or upload a class picture. Faces are counted, you confirm, and the record is saved.</p></div>
  </div>
  <div class="clock"><span id="clock"></span><small>Counts faces only. Nobody is identified.</small></div>
</div></div>

<div class="wrap"><div class="stack">

  <!-- STEP 1 -->
  <section class="panel p-session">
    <header><div class="badge">1</div><div><h2>Session details</h2><p>Name the class so the record is easy to find later</p></div></header>
    <div class="body">
      <label class="f" for="cls">Class or session name</label>
      <input type="text" id="cls" value="CS101 Lecture" maxlength="60">

      <details>
        <summary>Detection settings (for testing and tuning)</summary>
        <div class="setgrid">
          <div>
            <div class="l"><label for="scale">Scale factor</label><output id="scaleO">1.10</output></div>
            <input type="range" id="scale" min="1.03" max="1.5" step="0.01" value="1.1">
            <p class="hint">Lower finds more faces but is slower.</p>
          </div>
          <div>
            <div class="l"><label for="neigh">Min neighbors</label><output id="neighO">5</output></div>
            <input type="range" id="neigh" min="1" max="12" step="1" value="5">
            <p class="hint">Raise it to remove false detections.</p>
          </div>
          <div>
            <div class="l"><label for="minsz">Min face size</label><output id="minszO">30 px</output></div>
            <input type="range" id="minsz" min="20" max="200" step="5" value="30">
            <p class="hint">Raise it to ignore tiny, far-away faces.</p>
          </div>
        </div>
        <div class="setfoot">
          <label class="chk"><input type="checkbox" id="boost"> Boost poor lighting</label>
          <label class="chk"><input type="checkbox" id="prof"> Include side-profile faces</label>
          <button class="btn ghost" id="reset" type="button">Reset to defaults</button>
        </div>
      </details>
    </div>
  </section>

  <!-- STEP 2: two options -->
  <div class="two">
    <section class="panel p-camera">
      <header><div class="badge">A</div><div><h2>Take a picture with the camera</h2><p>Uses your computer's webcam</p></div></header>
      <div class="body">
        <div class="screen" id="camBox">
          <video id="cam" playsinline muted hidden></video>
          <span class="recdot" id="recdot" hidden>Camera on</span>
          <div class="off" id="camOff"><b>Camera is off</b>Press Start camera, then allow access in your browser</div>
        </div>
        <div class="btns">
          <button class="btn" id="camStart" type="button">Start camera</button>
          <button class="btn ghost" id="camShot" type="button" disabled>Take picture</button>
          <button class="btn danger" id="camStop" type="button" disabled>Stop camera</button>
        </div>
      </div>
    </section>

    <section class="panel p-upload">
      <header><div class="badge">B</div><div><h2>Upload a picture</h2><p>Use a photo you already have</p></div></header>
      <div class="body">
        <div class="drop" id="drop" tabindex="0" role="button" aria-label="Choose a picture to upload">
          <svg width="52" height="52" viewBox="0 0 24 24" fill="none" stroke="#047857" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M12 16V4M7 9l5-5 5 5"/><path d="M4 16v3a1 1 0 0 0 1 1h14a1 1 0 0 0 1-1v-3"/></svg>
          <strong>Drag a picture here</strong>
          <span>JPG, PNG or WebP, up to 25 MB</span>
        </div>
        <input type="file" id="file" accept="image/*" hidden>
        <div class="btns"><button class="btn" id="pick" type="button">Choose picture</button></div>
      </div>
    </section>
  </div>

  <!-- STEP 3 -->
  <section class="panel p-result" id="resultCard">
    <header><div class="badge">3</div><div><h2>Check the result and save the attendance</h2><p>Compare the numbered boxes with the real students</p></div></header>
    <div class="body">
      <div class="resgrid">
        <div class="pic">
          <canvas id="result" hidden></canvas>
          <div class="none" id="resultEmpty"><b>No picture yet</b>Take a picture (A) or upload one (B) above</div>
        </div>
        <div class="sidebox">
          <div class="cap">Faces detected</div>
          <div class="big" id="detected" aria-live="polite">–</div>
          <div class="status" id="status"></div>
          <hr>
          <label class="f" for="confirmed">Students present (confirmed)</label>
          <input type="number" id="confirmed" min="0" max="1000" step="1" disabled>
          <p class="hint">If a face was missed or a box is wrong, correct the number.</p>
          <label class="f" for="note" style="margin-top:12px">Note (optional)</label>
          <input type="text" id="note" maxlength="200" placeholder="e.g. 2 students late">
          <button class="btn" id="saveBtn" type="button" disabled>Save attendance record</button>
          <button class="btn ghost" id="dlPic" type="button" disabled>Download annotated picture</button>
        </div>
      </div>
    </div>
  </section>

  <!-- STEP 4 -->
  <section class="panel p-records">
    <header><div class="badge">4</div><div><h2>Attendance records</h2><p>Every saved session, ready to download as CSV</p></div></header>
    <div class="body">
      <div class="stats">
        <div class="stat"><b id="sN">0</b><span>Records saved</span></div>
        <div class="stat"><b id="sLast">–</b><span>Last present count</span></div>
        <div class="stat"><b id="sAvg">–</b><span>Average present</span></div>
      </div>
      <div class="tablewrap">
        <table>
          <thead><tr><th>Date</th><th>Time</th><th>Class</th><th>Source</th><th>Detected</th><th class="pres">Present</th><th>Note</th></tr></thead>
          <tbody id="rows"></tbody>
        </table>
      </div>
      <div class="btns head">
        <a class="btn" href="/export" download>Download CSV file</a>
        <button class="btn danger" id="clear" type="button">Clear all records</button>
      </div>
    </div>
  </section>

</div></div>
<div id="toast" role="status"></div>

<script>
const $ = id => document.getElementById(id);
const S = {blob:null, source:"", img:null, boxes:[], w:0, h:0, count:null, stream:null};

/* ---------- small helpers ---------- */
let tt;
function toast(msg, kind=""){const t=$("toast");t.textContent=msg;t.className="show "+kind;clearTimeout(tt);tt=setTimeout(()=>t.className="",3200);}
function tick(){
  const d=new Date();
  $("clock").textContent=d.toLocaleDateString(undefined,{weekday:"long",day:"numeric",month:"long",year:"numeric"})+" · "+d.toLocaleTimeString([],{hour:"2-digit",minute:"2-digit"});
}
tick(); setInterval(tick, 15000);

/* ---------- settings ---------- */
const fmt={scale:v=>(+v).toFixed(2),neigh:v=>v,minsz:v=>v+" px"};
const pairs=[["scale","scaleO"],["neigh","neighO"],["minsz","minszO"]];
let deb; function rerun(){ if(S.blob){clearTimeout(deb);deb=setTimeout(()=>analyze(false),250);} }
pairs.forEach(([id,o])=>{
  $(id).addEventListener("input",()=>$(o).textContent=fmt[id]($(id).value));
  $(id).addEventListener("change",rerun);
});
["boost","prof"].forEach(id=>$(id).addEventListener("change",rerun));
$("reset").onclick=()=>{
  $("scale").value=1.1;$("neigh").value=5;$("minsz").value=30;$("boost").checked=false;$("prof").checked=false;
  pairs.forEach(([id,o])=>$(o).textContent=fmt[id]($(id).value)); rerun();
};

/* ---------- drawing ---------- */
function drawBoxes(ctx, boxes, sx, sy){
  const lw=Math.max(3,ctx.canvas.width/300), fs=Math.max(16,ctx.canvas.width/42);
  ctx.lineWidth=lw; ctx.font=`700 ${fs}px "Outfit", sans-serif`; ctx.textBaseline="middle";
  boxes.forEach(([x,y,w,h],i)=>{
    x*=sx;y*=sy;w*=sx;h*=sy;
    ctx.strokeStyle="#ffb000"; ctx.strokeRect(x,y,w,h);
    const t=String(i+1), tw=ctx.measureText(t).width+fs*0.9, th=fs*1.5, ty=y-th<0?y:y-th;
    ctx.fillStyle="#ffb000"; ctx.fillRect(x-lw/2,ty,tw,th);
    ctx.fillStyle="#1b1f3b"; ctx.fillText(t,x+fs*0.45-lw/2,ty+th/2);
  });
}
function render(){
  const c=$("result"), x=c.getContext("2d");
  x.drawImage(S.img,0,0,c.width,c.height);
  if(S.w) drawBoxes(x,S.boxes,c.width/S.w,c.height/S.h);
}

/* ---------- detection ---------- */
function processBlob(blob, source){
  const url=URL.createObjectURL(blob), img=new Image();
  img.onload=()=>{
    const s=Math.min(1,1280/img.naturalWidth), c=$("result");
    c.width=Math.round(img.naturalWidth*s); c.height=Math.round(img.naturalHeight*s);
    S.img=img;S.blob=blob;S.source=source;S.boxes=[];S.w=0;
    $("resultEmpty").hidden=true; c.hidden=false;
    render(); URL.revokeObjectURL(url); analyze(true);
  };
  img.onerror=()=>toast("Could not open that picture. Use JPG, PNG or WebP.","error");
  img.src=url;
}
function setCount(n){
  S.count=n; $("detected").textContent=n; $("confirmed").value=n; $("confirmed").disabled=false;
  $("saveBtn").disabled=false; $("saveBtn").textContent="Save attendance record"; $("dlPic").disabled=false;
}
async function analyze(scroll){
  if(!S.blob) return;
  $("status").textContent="Detecting faces…";
  const fd=new FormData(); fd.append("file",S.blob,"picture.jpg");
  fd.append("scale",$("scale").value); fd.append("neighbors",$("neigh").value); fd.append("min_size",$("minsz").value);
  fd.append("boost",$("boost").checked?"1":"0"); fd.append("profile",$("prof").checked?"1":"0");
  try{
    const r=await fetch("/detect",{method:"POST",body:fd}), d=await r.json();
    if(!r.ok) throw new Error(d.error||"Detection failed");
    S.boxes=d.boxes;S.w=d.width;S.h=d.height; setCount(d.count); render();
    $("status").textContent=d.count===0?"No faces found. Try boosting lighting or lowering min neighbors.":"";
    if(scroll) $("resultCard").scrollIntoView({behavior:"smooth",block:"start"});
  }catch(e){ toast(e.message,"error"); $("status").textContent=""; }
}

/* ---------- Option A: camera ---------- */
function camUI(on){
  $("cam").hidden=!on; $("recdot").hidden=!on; $("camOff").hidden=on;
  $("camStart").disabled=on; $("camShot").disabled=!on; $("camStop").disabled=!on;
}
$("camStart").onclick=async()=>{
  if(!navigator.mediaDevices||!navigator.mediaDevices.getUserMedia){
    toast("Camera not available. Open the page at http://localhost:5000","error"); return;
  }
  try{
    S.stream=await navigator.mediaDevices.getUserMedia({video:{width:{ideal:1280},height:{ideal:720}},audio:false});
  }catch(e){
    toast("Camera blocked. Click the lock icon in the address bar, allow the camera, and close other apps using it.","error"); return;
  }
  $("cam").srcObject=S.stream; await $("cam").play(); camUI(true);
};
function stopCamera(){
  if(S.stream){S.stream.getTracks().forEach(t=>t.stop());S.stream=null;}
  $("cam").srcObject=null; camUI(false);
}
$("camStop").onclick=stopCamera;
$("camShot").onclick=()=>{
  const v=$("cam"); if(!v.videoWidth) return;
  const s=Math.min(1,1280/v.videoWidth), c=document.createElement("canvas");
  c.width=Math.round(v.videoWidth*s); c.height=Math.round(v.videoHeight*s);
  c.getContext("2d").drawImage(v,0,0,c.width,c.height);
  const box=$("camBox"); box.classList.remove("flash"); void box.offsetWidth; box.classList.add("flash");
  c.toBlob(b=>processBlob(b,"Camera picture"),"image/jpeg",0.92);
};

/* ---------- Option B: upload ---------- */
function loadFile(f){
  if(!f) return;
  if(!f.type.startsWith("image/")){toast("Please choose a picture file (JPG, PNG or WebP)","error");return;}
  processBlob(f,f.name);
}
$("pick").onclick=()=>$("file").click();
$("drop").onclick=()=>$("file").click();
$("drop").onkeydown=e=>{if(e.key==="Enter"||e.key===" "){e.preventDefault();$("file").click();}};
$("file").onchange=e=>{loadFile(e.target.files[0]);e.target.value="";};
["dragenter","dragover"].forEach(ev=>$("drop").addEventListener(ev,e=>{e.preventDefault();$("drop").classList.add("over");}));
["dragleave","drop"].forEach(ev=>$("drop").addEventListener(ev,e=>{e.preventDefault();$("drop").classList.remove("over");}));
$("drop").addEventListener("drop",e=>loadFile(e.dataTransfer.files[0]));

/* ---------- save and download ---------- */
$("confirmed").addEventListener("input",()=>{$("saveBtn").disabled=false;$("saveBtn").textContent="Save attendance record";});
$("saveBtn").onclick=async()=>{
  const n=parseInt($("confirmed").value,10);
  if(!Number.isInteger(n)||n<0||n>1000){toast("Enter a whole number between 0 and 1000","error");return;}
  const r=await fetch("/save",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({
    cls:$("cls").value.trim()||"Untitled class", source:S.source, detected:S.count, confirmed:n, note:$("note").value.trim()
  })});
  if(r.ok){
    toast(`Saved: ${n} student${n===1?"":"s"} present`);
    $("saveBtn").disabled=true; $("saveBtn").textContent="Saved ✓"; $("note").value=""; loadRecords();
  } else { const d=await r.json().catch(()=>({})); toast(d.error||"Could not save the record","error"); }
};
$("dlPic").onclick=()=>{
  $("result").toBlob(b=>{
    const a=document.createElement("a"); a.href=URL.createObjectURL(b);
    a.download="attendance-"+Date.now()+".png"; a.click();
  },"image/png");
};

/* ---------- records ---------- */
async function loadRecords(){
  const d=await (await fetch("/records")).json(), rows=d.rows, tb=$("rows");
  tb.innerHTML="";
  if(!rows.length){
    const tr=document.createElement("tr"), td=document.createElement("td");
    td.colSpan=7; td.textContent="No records yet. Take or upload a picture, then press Save attendance record.";
    td.style.color="#5a6280"; tr.appendChild(td); tb.appendChild(tr);
  }
  for(const r of rows){
    const tr=document.createElement("tr");
    for(const k of ["date","time","class","source","detected","present","note"]){
      const td=document.createElement("td"); td.textContent=r[k]??"";
      if(k==="present") td.className="pres"; if(k==="note") td.className="note";
      tr.appendChild(td);
    }
    tb.appendChild(tr);
  }
  const nums=rows.map(r=>+r.present).filter(n=>!isNaN(n));
  $("sN").textContent=rows.length;
  $("sLast").textContent=nums.length?nums[0]:"–";
  $("sAvg").textContent=nums.length?(nums.reduce((a,b)=>a+b,0)/nums.length).toFixed(1):"–";
}
$("clear").onclick=async()=>{
  if(!confirm("Delete ALL attendance records? This cannot be undone.")) return;
  await fetch("/clear",{method:"POST"}); loadRecords(); toast("All records cleared");
};

window.addEventListener("beforeunload", stopCamera);
loadRecords();
</script>
</body>
</html>
"""

In [4]:
import threading

def start():
    app.run(port=5000, debug=False, use_reloader=False)

threading.Thread(target=start, daemon=True).start()
print("Open http://localhost:5000")

Open http://localhost:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [31/Oct/2026 22:34:18] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [31/Oct/2026 22:34:18] "GET /records HTTP/1.1" 200 -
127.0.0.1 - - [31/Oct/2026 22:34:25] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [31/Oct/2026 22:34:34] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [31/Oct/2026 22:34:42] "POST /detect HTTP/1.1" 200 -
127.0.0.1 - - [31/Oct/2026 22:34:47] "POST /save HTTP/1.1" 200 -
127.0.0.1 - - [31/Oct/2026 22:34:47] "GET /records HTTP/1.1" 200 -
